# 01 — Agent CLI Single-Turn com Tool-use

**Aluna:** Danielle Magalhães Ballester (02/06/2026)  

**Objetivo:** Usar Pydantic + retry + structured output do lab anterior para construir um agente com function-calling.

**O que será feito:**
1. Definir duas tools (calculadora + consulta de documentação) com schema JSON
2. Implementar o loop LLM→tool→LLM (single-turn agent)
3. Comparar pure-prompt vs tool-use no mesmo problema

Referência completa: `lab-001.pdf`

## Setup

**Provider:** Groq (API gratuita — sem rate limit agressivo para tool-use).  
**Como gerar a chave:** https://console.groq.com/keys

Usei o SDK `openai` — o Groq expõe endpoint OpenAI-compatible.  
O modelo default é `llama-3.3-70b-versatile` (suporta function-calling de forma confiável).

In [7]:
import json
import os
import random
import time
from typing import Any

from openai import OpenAI
from pydantic import BaseModel, Field

# --- Carregar GROQ_API_KEY conforme o ambiente ---
# Ordem de tentativa: Colab Secrets -> .env local -> getpass prompt.


def _load_groq_key() -> tuple[str, str]:
    try:
        from google.colab import userdata
        try:
            key = userdata.get("GROQ_API_KEY")
            if key:
                return key, "Colab Secrets"
        except Exception:
            pass
    except ImportError:
        pass

    try:
        from dotenv import find_dotenv, load_dotenv
        dotenv_path = find_dotenv(usecwd=True)
        if dotenv_path:
            load_dotenv(dotenv_path)
    except ImportError:
        pass
    key = os.getenv("GROQ_API_KEY")
    if key:
        return key, ".env local"

    from getpass import getpass
    key = getpass("Cole sua GROQ_API_KEY (gere em https://console.groq.com/keys): ")
    if not key:
        raise RuntimeError("GROQ_API_KEY nao fornecida — abortando.")
    return key, "prompt interativo"


api_key, key_source = _load_groq_key()

client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1",
)
MODEL = "llama-3.3-70b-versatile"

print(f"Provider: Groq ({MODEL})")
print(f"Key carregada de: {key_source}")

Cole sua GROQ_API_KEY (gere em https://console.groq.com/keys): ··········
Provider: Groq (llama-3.3-70b-versatile)
Key carregada de: prompt interativo


## Etapa 1 — Definir duas tools com schema JSON

Construção das duas ferramentas que o agente poderá usar:

| Tool | Descrição |
|---|---|
| `calculator` | Avalia expressões aritméticas simples (+, -, *, /, parênteses) com `eval` seguro |
| `lookup_doc` | Consulta um corpus local de documentação técnica por termo exato (case-insensitive) |

Cada tool precisa de:
1. Uma **função Python** executável
2. Um **schema JSON** (segundo a especificação OpenAI function-calling) que descreve nome, descrição e parâmetros
3. Um **registro** (`TOOL_REGISTRY`) que mapeia nomes para funções

In [8]:
def calculator(expression: str) -> str:
    allowed = set("0123456789+-*/(). ")
    if not all(c in allowed for c in expression):
        return "ERROR: expressão contém caracteres não permitidos"
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"ERROR: {e}"


DOCS = {
    "retry": "tentar novamente após um erro, esperando um tempo antes de repetir.",
    "pydantic": "biblioteca Python que valida dados com BaseModel e type hints.",
    "streaming": "receber tokens um a um enquanto o modelo gera, em vez de esperar a resposta completa.",
}


def lookup_doc(term: str) -> str:
    return DOCS.get(term.lower(), f"NOT_FOUND: termo '{term}' nao encontrado")


TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Avalia uma expressão aritmetica simples (apenas + - * / e parenteses).",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "Expressão matematica, ex: '12 * (3 + 4)'",
                    }
                },
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "lookup_doc",
            "description": "Consulta um termo na base de documentacao local (retry, pydantic, streaming).",
            "parameters": {
                "type": "object",
                "properties": {
                    "term": {
                        "type": "string",
                        "description": "Termo a buscar, ex: 'retry'",
                    }
                },
                "required": ["term"],
            },
        },
    },
]

TOOL_REGISTRY = {"calculator": calculator, "lookup_doc": lookup_doc}

print(f"calculator('12 * (3 + 4)'): {calculator('12 * (3 + 4)')}")
print(f"lookup_doc('retry'): {lookup_doc('retry')}")

calculator('12 * (3 + 4)'): 84
lookup_doc('retry'): tentar novamente apos um erro, esperando um tempo antes de repetir.


**Reflexão:** por que o `calculator` valida o input antes de chamar `eval`? Que tipo de ataque essa validação previne?

Resposta: `eval()` executa **qualquer código Python**. Sem validar, um usuario malicioso poderia passar `"__import__('os').system('rm -rf /')"`. A whitelist de caracteres permite apenas digitos, operadores, parenteses, ponto e espaco — prevenindo injecao de codigo.

## Etapa 2 — Loop Tool→LLM→Tool (Single-Turn Agent)

O coração do lab: um loop que orquestra LLM e ferramentas.

```
Usuario → pergunta → LLM (+ schemas) → tool_call? → executa tool → resultado → LLM → resposta final
```

O LLM recebe os schemas das tools via o parametro `tools=TOOLS`. Se precisar de uma ferramenta, ele responde com `tool_calls` em vez de texto. O agente entao executa a funcao local e devolve o resultado como mensagem `"role": "tool"`. Quando o LLM ja tem o que precisa, ele para de chamar tools e devolve a resposta em texto livre.

In [9]:
def run_agent(user_query: str, max_iters: int = 5) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "Voce é um assistente que pode usar tools. "
                "Use 'calculator' para contas e 'lookup_doc' para definicoes técnicas. "
                "Sempre cite a fonte quando usar lookup_doc."
            ),
        },
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iters):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
        )
        msg = response.choices[0].message
        messages.append(msg.model_dump(exclude_unset=True))

        if not msg.tool_calls:
            return msg.content or ""

        for call in msg.tool_calls:
            fn_name = call.function.name
            args = json.loads(call.function.arguments)
            result = TOOL_REGISTRY[fn_name](**args)
            print(f"  [tool] {fn_name}({args}) => {result}")
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": call.id,
                    "content": result,
                }
            )

    return "[ERROR] max iterations excedidas"


queries = [
    "Quanto é 47 * 13 + 200?",
    "O que é retry em chamadas HTTP?",
    "Calcule 25% de 480 e me explique o que é pydantic",
]

for q in queries:
    print(f"\nQuery: {q}")
    print(f"Resposta: {run_agent(q)}")


Query: Quanto é 47 * 13 + 200?
  [tool] calculator({'expression': '47 * 13 + 200'}) => 811
Resposta: O resultado da expressão é 811.

Query: O que é retry em chamadas HTTP?
  [tool] lookup_doc({'term': 'retry'}) => tentar novamente apos um erro, esperando um tempo antes de repetir.
Resposta: A função de retry em chamadas HTTP é um mecanismo que permite que um cliente retente uma requisição que falhou devido a problemas de rede ou servidor. Isso é feito para evitar que o cliente tenha que lidar manualmente com erros e retries, melhorando a robustez e a confiabilidade da comunicação. Quando uma requisição HTTP falha, o cliente pode aguardar um tempo antes de tentar novamente, esperando que o problema tenha sido resolvido. Essa abordagem é útil em cenários em que a conectividade de rede é instável ou quando o servidor está temporariamente indisponível.

Query: Calcule 25% de 480 e me explique o que é pydantic
  [tool] calculator({'expression': '0.25 * 480'}) => 120.0
  [tool] lookup_doc(

**Verificacao:** as três queries devem produzir respostas corretas:
- `47 * 13 + 200` = **811**
- `retry` → definicao de retry
- `25% de 480` = **120** + definicao de pydantic (exige **2 tool calls consecutivas** na mesma conversa)

Se `tool_calls` for `None` mesmo com `tool_choice="auto"`, reforce o system prompt com "Sempre use 'calculator' para QUALQUER calculo aritmético."

## Etapa 3 — Comparativo Pure-Prompt vs Tool-use

Agora vamos testar as mesmas 3 perguntas sem usar ferramentas. O modelo vai responder com o que sabe de cor.

In [10]:
def run_pure_prompt(user_query: str) -> str:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Responda diretamente. Faça contas mentalmente."},
            {"role": "user", "content": user_query},
        ],
    )
    return response.choices[0].message.content or ""


print("=== PURE PROMPT ===")
for q in queries:
    print(f"\nQuery: {q}")
    print(f"Resposta: {run_pure_prompt(q)}")

=== PURE PROMPT ===

Query: Quanto é 47 * 13 + 200?
Resposta: Vou fazer a conta:

47 * 13 = 611
611 + 200 = 811

A resposta é 811.

Query: O que é retry em chamadas HTTP?
Resposta: Retry em chamadas HTTP refere-se à capacidade de um cliente (normalmente um programa ou uma aplicação) reenviar uma requisição HTTP que falhou devido a um erro temporário ou intermitente. Isso é feito para garantir que a requisição seja processada com sucesso, mesmo que a primeira tentativa não tenha sido bem-sucedida.

Quando uma requisição HTTP é enviada, ela pode falhar por vários motivos, como:

1. **Conexão de rede instável**: A conexão de rede pode ser interrompida ou instável, levando a erros de conexão.
2. **Erro no servidor**: O servidor pode estar temporariamente indisponível ou sobrecarregado, retornando erros como 500 (Erro Interno do Servidor) ou 503 (Serviço Indisponível).
3. **Tempo limite**: A requisição pode expirar devido a um tempo limite fixado pelo cliente ou servidor.

Nesses casos, o r

### Comparação esperada

| Query | Pure-prompt | Tool-use | Recomendado |
|---|---|---|---|
| Aritmética com >2 digitos | erro silencioso ocasional | sempre exato | **Tool-use** |
| Definição factual | depende do que o modelo aprendeu durante o treinamento | ancorado em fonte | **Tool-use** |
| Conversa aberta | natural | overhead desnecessário | **Pure-prompt** |

**Reflexão:** compare as seis respostas (três com tool, três sem). Em qual delas o LLM puro errou ou alucinou? Em qual o tool-use foi overkill?

Tool-use é ideal quando **precisão importa** (contas, fatos verificáveis). Pure-prompt é suficiente para **conversa aberta** onde o custo de chamar uma tool supera o benefício.

## Verificação

- [ ] Schemas JSON estao definidos para `calculator` e `lookup_doc` com `parameters.required` populado
- [ ] `TOOL_REGISTRY` mapeia nomes para funções Python executáveis
- [ ] Loop tool→LLM→tool termina por resposta final ou por `max_iters`
- [ ] As três queries de teste produzem resultados corretos via tool-use
- [ ] O comparativo entre pure-prompt e tool-use está documentado em texto livre (no minimo três frases)


## Troubleshooting

**Problema 1. `tool_calls` e `None` mesmo com `tool_choice="auto"`**  
Sintoma: o LLM responde direto sem usar a tool, mesmo em conta complexa.  
Causa: system prompt fraco — o LLM nâo percebe que tem permissão para chamar tool.  
Solução: reforce a instrução com algo como "Sempre use 'calculator' para QUALQUER cálculo aritmético." Ou forçe via `tool_choice={"type": "function", "function": {"name": "calculator"}}`.

**Problema 2. `json.JSONDecodeError` em `call.function.arguments`**  
Sintoma: erro ao parsear os argumentos da tool call.  
Causa: o LLM gerou JSON inválido (raro com modelos novos, comum com modelos antigos).  
Solução: envolver o `json.loads` em `try/except` e devolver o erro como tool result (`"ERROR: invalid arguments"`). O LLM tipicamente faz auto-correção no próximo turno.

**Problema 3. Loop infinito (LLM chama a mesma tool repetidamente)**  
Sintoma: `max_iters` e excedido sem resposta final.  
Causa: uma tool retornou erro mas o LLM não interpretou e simplesmente repetiu a chamada.  
Solução: padronizar os resultados começando com `OK:` ou `ERROR:` e orientar no system prompt: "Se uma tool retornar ERROR, NÃO repita a mesma chamada; resuma o erro ao usuário."